In [ ]:
# =============================================================================
#  HIERARCHICAL MULTINOMIAL LOGIT — MIXTURE OF NORMALS  (v2)
#  Full pipeline: Simulation → Model → Separated Init → Inference →
#                 Label Correction → Diagnostics
# =============================================================================
 
# ── Imports ───────────────────────────────────────────────────────────────────
 
import liesel.model as lsl
import liesel.goose as gs
 
from tensorflow_probability.substrates.jax.experimental import distributions as tfde
from tensorflow_probability.python.internal.backend.jax.compat import v2 as tf
import tensorflow_probability.substrates.jax.distributions as tfd
import tensorflow_probability.substrates.jax.bijectors as tfb
 
import jax.numpy as jnp
import jax
 
import numpy as np
import pandas as pd
 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import seaborn as sns
 
import time
import datetime
import functools
 
from IPython.display import display
 
# ── Plotting defaults ─────────────────────────────────────────────────────────
 
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
# =============================================================================
#  1. DATA SIMULATION
# =============================================================================
 
def generate_mixture_simulated_data(n_units=300, n_obs=50, n_components=2, seed=42):
    """
    Generates synthetic Hierarchical Multinomial Logit (HMNL) choice data
    drawn from a mixture of K multivariate normals.
 
    Architecture
    ────────────
    • n_units consumers, each making n_obs repeated discrete choices
      among 4 alternatives.
    • Consumers belong (latently) to one of K=2 segments with mixing weights π.
    • Segment-level preference means are driven by demographics via Delta[k].
    • Individual preferences β_i are drawn from their segment's MVN distribution.
    • Observed choices follow a Categorical(softmax(X @ β_i)) likelihood.
 
    Parameters
    ──────────
    n_units      : number of individuals
    n_obs        : choice tasks per individual
    n_components : K, number of mixture components
    seed         : random seed for reproducibility
 
    Returns
    ───────
    dict with JAX arrays (X, y, Z, unit_idx) and ground-truth parameters.
    """
    np.random.seed(seed)
 
    n_alts   = 4
    n_params = 4   # Brand 1 ASC, Brand 2 ASC, Brand 3 ASC, Price
    n_demos  = 2   # Intercept + one continuous demographic
 
    # ── Demographics ──────────────────────────────────────────────────────────
    Z = np.column_stack([np.ones(n_units), np.random.normal(0, 1, n_units)])
 
    # ── True mixing weights ───────────────────────────────────────────────────
    pi_true = np.array([0.6, 0.4]) if n_components == 2 \
              else np.ones(n_components) / n_components
 
    # ── True segment-level global parameters Delta[k] ─────────────────────────
    # Shape: (K, n_demos, n_params)
    Delta_true = np.zeros((n_components, n_demos, n_params))
 
    # Segment 0: strong Brand 1 preference, price-sensitive
    Delta_true[0] = np.array([
        [ 2.5, -0.5, -0.5, -2.0],   # intercept row
        [ 0.8, -0.3,  0.0,  0.5],   # demographic effect row
    ])
 
    if n_components > 1:
        # Segment 1: strong Brand 3 preference, less price-sensitive
        Delta_true[1] = np.array([
            [-1.5, -0.5,  2.5, -0.5],
            [-0.5,  0.2,  0.5, -0.1],
        ])
 
    # ── True within-segment covariance ────────────────────────────────────────
    # Shape: (K, n_params, n_params)
    Sigma_true = np.stack([np.diag([1.0, 0.5, 1.5, 0.5])] * n_components)
 
    # ── Individual-level parameters ───────────────────────────────────────────
    beta_true             = np.zeros((n_units, n_params))
    component_assignments = np.random.choice(n_components, size=n_units, p=pi_true)
 
    for i in range(n_units):
        k         = component_assignments[i]
        mu_i      = Z[i] @ Delta_true[k]
        beta_true[i] = np.random.multivariate_normal(mu_i, Sigma_true[k])
 
    # ── Simulate choices ──────────────────────────────────────────────────────
    X_list, y_list, unit_idx_list = [], [], []
 
    for i in range(n_units):
        for t in range(n_obs):
            X_it = np.zeros((n_alts, n_params))
            X_it[1, 0] = 1.0; X_it[2, 1] = 1.0; X_it[3, 2] = 1.0
            X_it[:, 3] = np.random.uniform(1.0, 5.0, n_alts)
 
            U_it  = X_it @ beta_true[i]
            probs = np.exp(U_it) / np.sum(np.exp(U_it))
            y_it  = np.random.choice(n_alts, p=probs)
 
            X_list.append(X_it)
            y_list.append(y_it)
            unit_idx_list.append(i)
 
    return {
        "X":               jnp.array(X_list),
        "y":               jnp.array(y_list),
        "Z":               jnp.array(Z),
        "unit_idx":        jnp.array(unit_idx_list),
        "n_units":         n_units,
        "n_params":        n_params,
        "n_alts":          n_alts,
        "n_z":             n_demos,
        "TRUE_DELTA":      Delta_true,
        "TRUE_SIGMA":      Sigma_true,
        "TRUE_BETA":       beta_true,
        "TRUE_COMPONENTS": component_assignments,
        "TRUE_PI":         pi_true,
    }
 
 
# ── Configure simulation here ─────────────────────────────────────────────────
K_COMPONENTS = 2
 
print("Simulating mixture data …")
choice_data = generate_mixture_simulated_data(
    n_units=300,
    n_obs=50,
    n_components=K_COMPONENTS,
    seed=42,
)
 
n_params    = choice_data["n_params"]
param_names = ["Brand 1", "Brand 2", "Brand 3", "Price"]
demo_names  = ["Base (Intercept)", "Demographic 1"]
 
print(f"  Units : {choice_data['n_units']}")
print(f"  Obs   : {choice_data['y'].shape[0]}")
print(f"  K     : {K_COMPONENTS}")
print(f"  True π: {choice_data['TRUE_PI']}")

In [ ]:
# =============================================================================
#  2. DISTRIBUTION HELPERS
# =============================================================================
 
def make_wishart(df, scale_tril):
    """Wishart prior over the Cholesky of the precision matrix."""
    return tfd.WishartTriL(
        df=df, scale_tril=scale_tril,
        input_output_cholesky=True, validate_args=False,
    )
 
 
def make_delta_prior(precision_factors, K, n_demos, n_params):
    """
    Matrix-normal-like prior for Delta (Zellner g-prior style).
 
    The precision is shared across components and scaled by the
    population-level covariance, encouraging shrinkage toward zero
    while adapting to the scale of individual heterogeneity.
    """
    prec_b = jnp.broadcast_to(
        precision_factors[:, None, :, :], (K, n_demos, n_params, n_params)
    )
    return tfde.MultivariateNormalPrecisionFactorLinearOperator(
        loc=jnp.zeros(n_params),
        precision_factor=tf.linalg.LinearOperatorLowerTriangular(prec_b),
        validate_args=False,
    )
 
 
def make_mixture(locs, precision_factors, logits, n_units, K, n_params):
    """
    Mixture-of-normals distribution for individual-level β_i.
 
    Each consumer's parameters are drawn from a K-component mixture whose
    means are Z @ Delta[k] and whose precision matrices come from Sigma_inv[k].
    """
    prec_b = jnp.broadcast_to(
        precision_factors[None, ...], (n_units, K, n_params, n_params)
    )
    components = tfde.MultivariateNormalPrecisionFactorLinearOperator(
        loc=locs,
        precision_factor=tf.linalg.LinearOperatorLowerTriangular(prec_b),
        validate_args=False,
    )
    mixture = tfd.Categorical(logits=jnp.broadcast_to(logits, (n_units, K)))
    return tfd.MixtureSameFamily(
        mixture_distribution=mixture,
        components_distribution=components,
    )

In [ ]:
# =============================================================================
#  3. MODEL BUILDING
# =============================================================================
 
def build_mixture_hmnl_model(data_dict, K=2, A=0.01):
    """
    Assembles the full Liesel probabilistic graphical model.
 
    Parameter hierarchy
    ───────────────────
    w_logits          ~ Normal(0, 5)              [K]          mixing weights
    sigma_inv_chol    ~ WishartTriL(ν, V⁻¹)      [K, P, P]    precision Cholesky
    Delta             ~ MatrixNormal(0, Σ/A)      [K, D, P]    global parameters
    β_i               ~ MixtureNormal(…)          [N, P]       individual params
    y                 ~ Categorical(softmax(Xβ))               observed choices
 
    Parameters
    ──────────
    data_dict : output of generate_mixture_simulated_data()
    K         : number of mixture components
    A         : g-prior scaling constant (smaller = stronger shrinkage)
    """
    n_params = int(data_dict["n_params"])
    n_units  = int(data_dict["n_units"])
    n_demos  = data_dict["Z"].shape[1]
 
    # ── Mixture weights ───────────────────────────────────────────────────────
    w_logits = lsl.Var.new_param(
        value=jnp.zeros(K),
        distribution=lsl.Dist(tfd.Normal, loc=0., scale=5.),
        name="w_logits",
    )
 
    # ── Wishart prior for precision matrices ──────────────────────────────────
    nu        = float(n_params + 3)
    Vinv_chol = jnp.linalg.cholesky(jnp.linalg.inv(nu * jnp.eye(n_params)))
 
    sigma_inv_chol = lsl.Var.new_param(
        value=jnp.stack([jnp.eye(n_params)] * K),
        distribution=lsl.Dist(
            functools.partial(make_wishart, df=nu),
            scale_tril=jnp.stack([Vinv_chol] * K),
        ),
        name="sigma_inv_chol",
    )
 
    # Latent parameterisation for efficient NUTS sampling
    sigma_inv_chol_latent = sigma_inv_chol.transform(
        tfb.FillScaleTriL(), name="sigma_inv_chol_latent"
    )
 
    # Precision factor for the Delta prior
    mu_prec_factor = lsl.Var.new_calc(
        lambda L: jnp.sqrt(A) * L,
        L=sigma_inv_chol,
        name="mu_prec_factor",
    )
 
    # ── Global parameters Delta ───────────────────────────────────────────────
    Z_var = lsl.Var.new_obs(data_dict["Z"], name="Z_obs")
 
    global_param = lsl.Var.new_param(
        value=jnp.zeros((K, n_demos, n_params)),
        distribution=lsl.Dist(
            functools.partial(
                make_delta_prior, K=K, n_demos=n_demos, n_params=n_params
            ),
            precision_factors=mu_prec_factor,
        ),
        name="Delta",
    )
 
    # Individual-level prior means: Z @ Delta[k]  →  shape (n_units, K, n_params)
    loc_var = lsl.Var.new_calc(
        lambda z, delta: jnp.einsum("ud,kdp->ukp", z, delta),
        z=Z_var, delta=global_param,
        name="beta_loc",
    )
 
    # ── Individual-level parameters ───────────────────────────────────────────
    beta_i = lsl.Var.new_param(
        value=jnp.zeros((n_units, n_params)),
        distribution=lsl.Dist(
            functools.partial(
                make_mixture, n_units=n_units, K=K, n_params=n_params
            ),
            locs=loc_var,
            precision_factors=sigma_inv_chol,
            logits=w_logits,
        ),
        name="beta_i",
    )
 
    # ── Likelihood ────────────────────────────────────────────────────────────
    X_var   = lsl.Var.new_obs(data_dict["X"],        name="X_obs")
    idx_var = lsl.Var.new_obs(data_dict["unit_idx"], name="idx_obs")
 
    beta_expanded = lsl.Var.new_calc(
        lambda b, idx: b[idx], b=beta_i, idx=idx_var, name="beta_expanded"
    )
    logits_var = lsl.Var.new_calc(
        lambda x, b: jnp.einsum("nij,nj->ni", x, b),
        x=X_var, b=beta_expanded, name="logits",
    )
    y_var = lsl.Var.new_obs(
        data_dict["y"],
        distribution=lsl.Dist(tfd.Categorical, logits=logits_var),
        name="y",
    )
 
    return lsl.Model([y_var])

In [ ]:
# =============================================================================
#  4. SEPARATED INITIALISATION
# =============================================================================
 
def build_separated_initial_state(model, K, n_params, n_demos,
                                   param_high=0, param_low=2,
                                   separation=2.0):
    """
    Builds a model state where Delta is initialised so that all K components
    are clearly separated on the pivot contrast from draw 0.
 
    Because NUTS is a local sampler it almost never crosses a large mode
    boundary once warmup begins. Starting all chains in the same canonical
    labeling therefore prevents chain-level label switching entirely,
    eliminating the 3-to-1 (or 2-to-2) chain splits.
 
    Initialisation strategy
    ───────────────────────
    Component 0 :  Delta[0, 0, param_high] = +separation
                   Delta[0, 0, param_low]  = -separation
    Component 1 :  Delta[1, 0, param_high] = -separation
                   Delta[1, 0, param_low]  = +separation
 
    All other Delta entries start at 0.  The sampler is free to move them.
 
    This mirrors the contrast used in relabel_by_contrast(), so both
    mechanisms agree on the canonical component ordering.
 
    Parameters
    ──────────
    model      : compiled lsl.Model
    K          : number of mixture components
    n_params   : number of preference parameters
    n_demos    : number of demographic variables (including intercept)
    param_high : parameter index with high value in Component 0
                 (default 0 = Brand 1 intercept)
    param_low  : parameter index with low  value in Component 0
                 (default 2 = Brand 3 intercept)
    separation : magnitude of the initial contrast.
                 2.0–3.0 works well for typical HMNL problems.
                 Increase to 3.0 if chains still occasionally flip.
 
    Returns
    ───────
    dict : model state with Delta_value overwritten; all other nodes unchanged.
    """
    state      = dict(model.state)
    delta_init = np.zeros((K, n_demos, n_params))
 
    for k in range(K):
        sign = 1.0 if k == 0 else -1.0
        delta_init[k, 0, param_high] =  sign * separation
        delta_init[k, 0, param_low]  = -sign * separation
 
    state["Delta_value"] = jnp.array(delta_init)
    return state

In [ ]:
# =============================================================================
#  5. INFERENCE
# =============================================================================
 
def run_mixture_inference(model, data_dict, K, n_params,
                          chains=4, warmup=1000, posterior=2000, seed=123,
                          separation=2.0):
    """
    Runs NUTS sampling via Goose with component-separated initialisation.
 
    All chains are started at the same separated Delta values so that
    the sampler's locality prevents inter-chain label switching.
 
    Four separate NUTS kernels allow each parameter block to adapt its
    own step size and mass matrix independently — important because the
    curvature differs substantially across (sigma, Delta, beta_i, w_logits).
 
    Parameters
    ──────────
    separation : passed to build_separated_initial_state().
                 Increase to 3.0 if the chain-alignment check still
                 shows splits after warmup.
    """
    n_demos = data_dict["Z"].shape[1]
 
    eb = gs.EngineBuilder(seed=seed, num_chains=chains)
    eb.set_model(gs.LieselInterface(model))
 
    init_state = build_separated_initial_state(
        model, K=K, n_params=n_params, n_demos=n_demos,
        param_high=0, param_low=2, separation=separation,
    )
    eb.set_initial_values(init_state)
 
    eb.add_kernel(gs.NUTSKernel(["sigma_inv_chol_latent"], mm_diag=False))
    eb.add_kernel(gs.NUTSKernel(["Delta"]))
    eb.add_kernel(gs.NUTSKernel(["beta_i"]))
    eb.add_kernel(gs.NUTSKernel(["w_logits"]))
 
    eb.set_duration(warmup_duration=warmup, posterior_duration=posterior)
 
    print("Starting NUTS sampling with separated initialisation …")
    engine = eb.build()
    engine.sample_all_epochs()
 
    results = engine.get_results()
    return results, results.get_posterior_samples()
 
 
print("\nBuilding model …")
mix_model = build_mixture_hmnl_model(choice_data, K=K_COMPONENTS)
 
start_time = time.time()
mcmc_results, posterior_samples = run_mixture_inference(
    mix_model, choice_data,
    K=K_COMPONENTS, n_params=n_params,
    chains=4, warmup=1000, posterior=2000,
    seed=123,
    separation=2.0,    # increase to 3.0 if chain-alignment check fails
)
print(f"Sampling finished in "
      f"{datetime.timedelta(seconds=int(time.time() - start_time))}")

In [ ]:
# =============================================================================
#  6. CHAIN-ALIGNMENT CHECK
# =============================================================================
 
def check_chain_alignment(posterior_samples, K,
                           pivot_demo=0, pivot_param=0,
                           agreement_threshold=1.0):
    """
    Verifies that all chains settled in the same labeling after warmup.
 
    Prints the per-chain posterior mean of the pivot parameter for each
    component. A large spread across chains on the same component index
    indicates that some chains flipped labels — the 3-to-1 problem.
 
    Parameters
    ──────────
    pivot_demo  : demographic row of the pivot (default 0 = intercept)
    pivot_param : parameter column of the pivot (default 0 = Brand 1)
    agreement_threshold : maximum acceptable spread in posterior means
                          across chains for the same component (default 1.0)
    """
    raw      = np.array(posterior_samples["Delta"])  # (C, D, K, demos, params)
    n_chains = raw.shape[0]
 
    print("\n=== Chain Alignment Check ===")
    print(f"  Pivot: Delta[k, demo={pivot_demo}, param={pivot_param}]")
    print()
    print(f"{'Chain':<8}", end="")
    for k in range(K):
        print(f"  Comp {k} mean", end="")
    print()
    print("─" * (8 + 14 * K))
 
    for c in range(n_chains):
        print(f"  {c:<6}", end="")
        for k in range(K):
            mean = np.mean(raw[c, :, k, pivot_demo, pivot_param])
            print(f"  {mean:>12.3f}", end="")
        print()
 
    # Spread check: all chains should agree on Component 0 mean
    comp0_means = [np.mean(raw[c, :, 0, pivot_demo, pivot_param])
                   for c in range(n_chains)]
    spread = np.max(comp0_means) - np.min(comp0_means)
    print()
    if spread < agreement_threshold:
        print(f"  ✓ All chains consistent  (spread = {spread:.3f})")
    else:
        print(f"  ✗ Chains DISAGREE  (spread = {spread:.3f})")
        print(f"    → Re-run with separation=3.0 in run_mixture_inference()")
 
 
check_chain_alignment(posterior_samples, K=K_COMPONENTS)

In [ ]:
# =============================================================================
#  7. LABEL-SWITCHING CORRECTION  (post-hoc safety net)
# =============================================================================
 
def relabel_by_contrast(posterior_samples, K, demo=0,
                         param_high=0, param_low=2):
    """
    Post-hoc label switching fix via a contrast ordering constraint.
 
    For each posterior draw the K component indices are permuted so that
        Delta[0, demo, param_high] - Delta[0, demo, param_low]
    is descending across components.
 
    Even with separated initialisation this is kept as a safety net —
    it catches any residual draw-level switching at zero extra cost.
 
    Parameters
    ──────────
    param_high : parameter with high value in Component 0 (default 0 = Brand 1)
    param_low  : parameter with low  value in Component 0 (default 2 = Brand 3)
 
    Returns
    ───────
    dict with same keys; component axes relabeled per draw.
    Shapes: (total_draws, K, …) — chains and draws are flattened.
    """
    delta  = np.array(posterior_samples["Delta"])
    sigma  = np.array(posterior_samples["sigma_inv_chol_latent"])
    logits = np.array(posterior_samples["w_logits"])
    beta   = np.array(posterior_samples["beta_i"])
 
    C, D, K_actual, n_demos, n_params_d = delta.shape
 
    delta_f  = delta.reshape(-1, K_actual, n_demos, n_params_d)
    sigma_f  = sigma.reshape(-1, K_actual, sigma.shape[-1])
    logits_f = logits.reshape(-1, K_actual)
    beta_f   = beta.reshape(-1, *beta.shape[2:])
    N        = delta_f.shape[0]
 
    contrast = (delta_f[:, :, demo, param_high]
              - delta_f[:, :, demo, param_low])
    perms    = np.argsort(contrast, axis=1)[:, ::-1]   # descending
 
    row_idx = np.arange(N)[:, None]
    return {
        "Delta":                 delta_f[row_idx, perms],
        "sigma_inv_chol_latent": sigma_f[row_idx, perms],
        "w_logits":              logits_f[row_idx, perms],
        "beta_i":                beta_f,
    }
 
 
def check_relabeling(relabeled_samples, true_delta_all, K,
                     pivot_demo=0, pivot_param=0):
    """
    Sanity check: posterior mean of the pivot vs true value per component.
    All components should show ✓.
    """
    delta_rl = relabeled_samples["Delta"]
    print("\n=== Relabeling Sanity Check ===")
    print(f"{'Component':<12} {'Post. Mean':>12} {'True Value':>12} {'Match?':>8}")
    print("─" * 50)
    for k in range(K):
        post_mean = np.mean(delta_rl[:, k, pivot_demo, pivot_param])
        true_val  = true_delta_all[k, pivot_demo, pivot_param]
        dists     = [abs(post_mean - true_delta_all[j, pivot_demo, pivot_param])
                     for j in range(K)]
        match     = "✓" if np.argmin(dists) == k else "✗  ← still misaligned"
        print(f"  Comp {k}      {post_mean:>12.3f} {true_val:>12.3f} {match:>8}")
 
 
print("\nApplying post-hoc label switching correction …")
rs = relabel_by_contrast(
    posterior_samples, K=K_COMPONENTS,
    demo=0, param_high=0, param_low=2,
)
check_relabeling(
    rs, choice_data["TRUE_DELTA"],
    K=K_COMPONENTS, pivot_demo=0, pivot_param=0,
)
 
# ── Convenience aliases ───────────────────────────────────────────────────────
delta_rl  = rs["Delta"]                         # (N, K, demos, params)
logits_rl = rs["w_logits"]                      # (N, K)
pi_rl     = jax.nn.softmax(logits_rl, axis=-1)  # (N, K)
beta_rl   = rs["beta_i"]                        # (N, n_units, n_params)
 
# ── Recover Σ from latent Cholesky ────────────────────────────────────────────
bijector = tfb.FillScaleTriL()
 
def latent_to_sigma(v):
    L = bijector.forward(v)
    return jnp.linalg.inv(L @ L.T)
 
# vmap over draws then over components
v_latent_to_sigma = jax.vmap(jax.vmap(latent_to_sigma))
 
sigma_rl = np.array(
    jax.vmap(v_latent_to_sigma)(
        rs["sigma_inv_chol_latent"].reshape(
            1, -1, K_COMPONENTS, rs["sigma_inv_chol_latent"].shape[-1]
        )
    )
).reshape(-1, K_COMPONENTS, n_params, n_params)  # (N, K, P, P)
 
N_draws = delta_rl.shape[0]
print(f"\nRelabeled sample shapes:")
print(f"  Delta  : {delta_rl.shape}   (draws, K, demos, params)")
print(f"  Pi     : {pi_rl.shape}      (draws, K)")
print(f"  Sigma  : {sigma_rl.shape}   (draws, K, params, params)")
print(f"  beta_i : {beta_rl.shape}    (draws, units, params)")

In [ ]:
# =============================================================================
#  8. SHARED HELPER FUNCTIONS
# =============================================================================
 
def compute_acf(x, nlags=30):
    """Autocorrelation function for a 1-D array."""
    x_c  = x - np.mean(x)
    norm = np.sum(x_c ** 2)
    if norm == 0:
        return np.zeros(nlags)
    acf = np.correlate(x_c, x_c, mode="full")
    return (acf[acf.size // 2:] / norm)[:nlags]
 
 
def generate_delta_table(delta_draws_Nkdp, k, p_names, d_names):
    """Returns a 'mean (std)' DataFrame for component k."""
    s    = delta_draws_Nkdp[:, k]
    mean = np.mean(s, axis=0)
    std  = np.std(s,  axis=0)
    df   = pd.DataFrame(index=d_names, columns=p_names)
    for i in range(len(d_names)):
        for j in range(len(p_names)):
            df.iloc[i, j] = f"{mean[i,j]: .2f} ({std[i,j]:.2f})"
    return df
 
 
def generate_delta_diff_table(true_delta_k, delta_draws_Nkdp, k, p_names, d_names):
    """Returns |True − Posterior Mean| DataFrame for component k."""
    diff = np.abs(true_delta_k - np.mean(delta_draws_Nkdp[:, k], axis=0))
    df   = pd.DataFrame(index=d_names, columns=p_names)
    for i in range(len(d_names)):
        for j in range(len(p_names)):
            df.iloc[i, j] = f"{diff[i,j]:.3f}"
    return df

In [ ]:
# =============================================================================
#  9. MIXTURE WEIGHTS DIAGNOSTICS
# =============================================================================
 
print("\n=== Mixture Weights (π) ===")
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Mixture Component Weights ($\\pi$) — After Relabeling", fontsize=15)
 
for k in range(K_COMPONENTS):
    true_pi = choice_data["TRUE_PI"][k]
    axes[0].plot(pi_rl[:, k], color=f"C{k}", alpha=0.4,
                 label=f"Comp {k} (True={true_pi:.2f})")
    sns.kdeplot(pi_rl[:, k], ax=axes[1], fill=True,
                color=f"C{k}", alpha=0.4,
                label=f"Comp {k} (True={true_pi:.2f})")
    axes[1].axvline(true_pi, color=f"C{k}", linestyle="--", lw=2)
 
axes[0].set_title("Trace of $\\pi_k$ (all chains pooled)")
axes[0].set_xlabel("Draw"); axes[0].set_ylabel("Probability"); axes[0].legend()
axes[1].set_title("Posterior Distribution of $\\pi_k$")
axes[1].set_xlabel("Probability"); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
#  10. CHOLESKY TRACE PLOTS — PER COMPONENT
# =============================================================================
 
def plot_cholesky_traces(latent_arr_CdKl, k, n_params, figsize=(15, 12)):
    """
    MCMC trace grid (lower-triangular layout) for the latent Cholesky
    of component k.
 
    Uses the raw (chains, draws, K, n_latent) array so per-chain
    colour separation is preserved in the trace plot.
    """
    n_chains, n_draws, K_actual, n_latent = latent_arr_CdKl.shape
 
    fig, axes = plt.subplots(n_params, n_params, figsize=figsize, sharex=True)
    if n_params == 1:
        axes = np.array([[axes]])
 
    latent_idx = 0
    for i in range(n_params):
        for j in range(n_params):
            ax = axes[i, j]
            if i >= j and latent_idx < n_latent:
                for chain in range(n_chains):
                    ax.plot(latent_arr_CdKl[chain, :, k, latent_idx],
                            label=f"Chain {chain}")
                ax.set_title(f"L[{i},{j}]", fontsize=10)
                latent_idx += 1
            else:
                ax.axis("off")
            ax.grid(True)
            if j == 0:
                ax.set_ylabel("Value")
 
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center",
               ncol=n_chains, bbox_to_anchor=(0.5, 0.02))
    plt.suptitle(
        f"MCMC Trace: sigma_inv_chol_latent — Component {k} "
        f"({n_params}×{n_params})",
        fontsize=16,
    )
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.show()
 
 
print("\n=== Cholesky Trace Plots ===")
raw_latent = np.array(posterior_samples["sigma_inv_chol_latent"])
 
for k in range(K_COMPONENTS):
    plot_cholesky_traces(raw_latent, k=k, n_params=n_params)
 
 
# =============================================================================
#  11. GOOSE-STYLE DELTA DIAGNOSTICS — PER COMPONENT
# =============================================================================
 
def plot_goose_style_delta(delta_arr_CdKdp, k, demo_idx, param_idx,
                            demo_name, param_name, n_lags=30):
    """
    3-panel Goose-style diagnostic (Trace / Density / ACF) for
    Delta[k, demo_idx, param_idx].
 
    Uses the raw (chains, draws, K, demos, params) array so each
    chain is plotted in its own colour.
    """
    n_chains = delta_arr_CdKdp.shape[0]
 
    fig = plt.figure(figsize=(12, 7))
    fig.suptitle(
        f"Component {k}  |  Delta[{demo_idx}, {param_idx}]"
        f"  ({demo_name} → {param_name})",
        fontsize=14, y=0.97,
    )
    gs_lay = gridspec.GridSpec(2, 2, height_ratios=[1.2, 1],
                                hspace=0.35, wspace=0.25)
 
    ax_trace = fig.add_subplot(gs_lay[0, :])
    for chain in range(n_chains):
        ax_trace.plot(delta_arr_CdKdp[chain, :, k, demo_idx, param_idx],
                      label=f"{chain}")
    ax_trace.set_xlabel("Iteration"); ax_trace.set_ylabel("Value")
    ax_trace.grid(True)
    ax_trace.legend(title="Chain", loc="center left",
                    bbox_to_anchor=(1.02, 0.5), frameon=False)
 
    ax_dens = fig.add_subplot(gs_lay[1, 0])
    for chain in range(n_chains):
        sns.kdeplot(delta_arr_CdKdp[chain, :, k, demo_idx, param_idx],
                    ax=ax_dens, fill=False)
    ax_dens.set_xlabel("Value"); ax_dens.set_ylabel("Density"); ax_dens.grid(True)
 
    ax_acf = fig.add_subplot(gs_lay[1, 1])
    for chain in range(n_chains):
        ax_acf.plot(compute_acf(
            delta_arr_CdKdp[chain, :, k, demo_idx, param_idx], nlags=n_lags
        ))
    ax_acf.set_xlabel("Lag"); ax_acf.set_ylabel("Autocorrelation")
    ax_acf.set_ylim(-0.1, 1.05); ax_acf.grid(True)
 
    plt.show()
 
 
print("\n=== Goose-Style Delta Diagnostics (Raw Chains) ===")
raw_delta = np.array(posterior_samples["Delta"])   # (C, D, K, demos, params)
 
for k in range(K_COMPONENTS):
    print(f"\n  Component {k}")
    for i in range(len(demo_names)):
        for j in range(len(param_names)):
            plot_goose_style_delta(
                raw_delta, k=k,
                demo_idx=i, param_idx=j,
                demo_name=demo_names[i],
                param_name=param_names[j],
            )

In [ ]:
# =============================================================================
#  12. DELTA SUMMARY TABLES — PER COMPONENT  (relabeled)
# =============================================================================
 
print("\n=== Global Parameter (Delta) Recovery — Per Component ===")
 
for k in range(K_COMPONENTS):
    true_delta_k = choice_data["TRUE_DELTA"][k]
    print(f"\n── Component {k} ──────────────────────────────────────")
 
    print("True Delta:")
    display(pd.DataFrame(true_delta_k, index=demo_names, columns=param_names))
 
    print("Posterior Delta  [mean (std)]:")
    display(generate_delta_table(delta_rl, k, param_names, demo_names))
 
    print("|True − Posterior Mean|:")
    display(generate_delta_diff_table(
        true_delta_k, delta_rl, k, param_names, demo_names
    ))

In [ ]:
# =============================================================================
#  13. DELTA POSTERIOR DISTRIBUTIONS — PER COMPONENT  (relabeled)
# =============================================================================
 
def plot_delta_distributions(delta_draws_Nkdp, true_delta_all, K,
                              p_names, d_names):
    """
    One figure per component: KDE posteriors vs true values for every
    (demographic × parameter) cell.
    """
    for k in range(K):
        fig, axes = plt.subplots(
            len(d_names), len(p_names),
            figsize=(4 * len(p_names), 3.5 * len(d_names)),
        )
        if len(d_names) == 1:
            axes = axes[np.newaxis, :]
 
        for d in range(len(d_names)):
            for p in range(len(p_names)):
                ax      = axes[d, p]
                samples = delta_draws_Nkdp[:, k, d, p]
                t_val   = true_delta_all[k, d, p]
                ci_lo, ci_hi = np.percentile(samples, [2.5, 97.5])
 
                sns.kdeplot(samples, ax=ax, fill=True,
                            color="#1f77b4", alpha=0.5, label="Posterior")
                ax.axvline(ci_lo,  color="#1f77b4", ls=":",  lw=1.5, label="95% CI")
                ax.axvline(ci_hi,  color="#1f77b4", ls=":",  lw=1.5)
                ax.axvline(t_val,  color="#D62728", ls="--", lw=2,
                           label="True Value")
 
                if d == 0: ax.set_title(p_names[p], fontweight="bold")
                if p == 0: ax.set_ylabel(d_names[d], fontweight="bold")
                ax.grid(True, alpha=0.3)
 
                if d == 0 and p == 0:
                    hl = ax.get_legend_handles_labels()
                    unique = [(h, l) for idx, (h, l)
                              in enumerate(zip(*hl)) if l not in hl[1][:idx]]
                    ax.legend(*zip(*unique), loc="upper right", fontsize=8)
 
        plt.suptitle(
            f"Posterior vs True — Delta (Component {k})",
            fontsize=16, y=1.02,
        )
        plt.tight_layout()
        plt.show()
 
 
print("\n=== Delta Posterior Distributions ===")
plot_delta_distributions(
    delta_rl, choice_data["TRUE_DELTA"],
    K_COMPONENTS, param_names, demo_names,
)

In [ ]:
# =============================================================================
#  14. COVARIANCE MATRIX RECOVERY — PER COMPONENT  (relabeled)
# =============================================================================
 
def plot_covariance_recovery(sigma_draws_NKpp, true_sigma_all, K, p_names):
    """
    Lower-triangular grid of KDE posteriors for each Σ element,
    one figure per component.
 
    Overlaid vertical lines show:
      ─  posterior mean
      ⋯  95% credible interval
      -- true parameter value
 
    sigma_draws_NKpp : (total_draws, K, n_params, n_params)
    true_sigma_all   : (K, n_params, n_params)
    """
    n_dim          = len(p_names)
    diag_color     = "#002347"
    off_diag_color = "#4682B4"
    true_color     = "#D62728"
 
    for k in range(K):
        samples_k = sigma_draws_NKpp[:, k]
        true_k    = true_sigma_all[k]
 
        fig, axes = plt.subplots(n_dim, n_dim, figsize=(18, 16))
 
        for i in range(n_dim):
            for j in range(n_dim):
                ax = axes[i, j]
                if j > i:
                    ax.axis("off")
                    continue
 
                data      = samples_k[:, i, j]
                color     = diag_color if i == j else off_diag_color
                pm        = np.mean(data)
                ci_lo, ci_hi = np.percentile(data, [2.5, 97.5])
                true_val  = true_k[i, j]
 
                sns.kdeplot(data, ax=ax, fill=True,
                            color=color, alpha=0.25, lw=2.5)
                ax.axvline(pm,       color=color,      lw=1.5, alpha=0.8)
                ax.axvline(ci_lo,    color=color,      ls=":", lw=1.8, alpha=0.7)
                ax.axvline(ci_hi,    color=color,      ls=":", lw=1.8, alpha=0.7)
                ax.axvline(true_val, color=true_color, ls="--", lw=2)
 
                ax.set_title(
                    f"Mean: {pm:.2f} | True: {true_val:.2f}\n"
                    f"95% CI: [{ci_lo:.2f}, {ci_hi:.2f}]",
                    fontsize=9, pad=10, linespacing=1.6, fontweight="bold",
                )
                ax.set_yticks([])
                sns.despine(ax=ax, left=True)
                if i == n_dim - 1:
                    ax.set_xlabel(p_names[j], fontsize=11, fontweight="bold")
                if j == 0:
                    ax.set_ylabel(p_names[i], fontsize=11, fontweight="bold")
 
        legend_elements = [
            Line2D([0], [0], color=off_diag_color, lw=2,
                   label="Posterior Mean"),
            Line2D([0], [0], color=off_diag_color, ls=":", lw=2,
                   label="95% Credible Interval"),
            Line2D([0], [0], color=true_color, ls="--", lw=2,
                   label="True Parameter Value"),
        ]
        fig.legend(handles=legend_elements, loc="upper right",
                   bbox_to_anchor=(0.9, 0.9), fontsize=13,
                   frameon=True, edgecolor="black", framealpha=1)
        plt.suptitle(
            f"Posterior Σ vs True Σ — Component {k}",
            fontsize=22, y=0.98, fontweight="light",
        )
        plt.subplots_adjust(hspace=0.6, wspace=0.2)
        plt.show()
 
 
print("\n=== Covariance Matrix Recovery ===")
plot_covariance_recovery(sigma_rl, choice_data["TRUE_SIGMA"], K_COMPONENTS, param_names)

In [ ]:
# =============================================================================
#  15. HOUSEHOLD β_i RECOVERY
# =============================================================================
 
print("\n=== Household-Level Parameter Recovery ===")
 
l_beta_means = np.mean(beta_rl, axis=0)   # (n_units, n_params)
true_betas   = choice_data["TRUE_BETA"]
 
# ── Scatter: posterior mean vs true ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
colors_p = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
 
for i, pname in enumerate(param_names):
    ax.scatter(true_betas[:, i], l_beta_means[:, i],
               alpha=0.6, s=25, label=pname, color=colors_p[i])
 
lo = min(true_betas.min(), l_beta_means.min())
hi = max(true_betas.max(), l_beta_means.max())
ax.plot([lo, hi], [lo, hi], "k--", lw=2, alpha=0.7, label="Perfect Recovery")
 
ax.set_title("Parameter Recovery: Posterior β Means vs True β", fontsize=16)
ax.set_xlabel("True Simulated β Values", fontsize=12)
ax.set_ylabel("Posterior Means", fontsize=12)
ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
plt.show()
 
 
# ── Per-household posterior distributions ─────────────────────────────────────
 
def plot_beta_distributions(samples_Np, true_vals_p, p_names, title,
                             color_post="#1f77b4", color_true="#d62728"):
    """
    2×2 KDE grid showing the posterior β distribution for a single
    household, with a vertical line at the true simulated value.
 
    samples_Np  : (total_draws, n_params)
    true_vals_p : (n_params,)
    """
    cols = 2
    rows = int(np.ceil(len(p_names) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
 
    for idx, (ax, pname) in enumerate(zip(axes.flatten(), p_names)):
        if idx >= len(p_names):
            ax.axis("off"); continue
 
        s            = samples_Np[:, idx]
        t            = true_vals_p[idx]
        ci_lo, ci_hi = np.percentile(s, [2.5, 97.5])
 
        sns.kdeplot(s, ax=ax, fill=True, color=color_post, alpha=0.5,
                    label="Posterior")
        ax.axvline(ci_lo, color=color_post, ls=":", lw=1.5, label="95% CI")
        ax.axvline(ci_hi, color=color_post, ls=":", lw=1.5)
        ax.axvline(t,     color=color_true, ls="--", lw=2,
                   label=f"True ({t:.2f})")
 
        ax.set_title(pname, fontsize=12, fontweight="bold")
        ax.grid(True, alpha=0.2)
 
        if idx == 0:
            hl = ax.get_legend_handles_labels()
            unique = [(h, l) for i_, (h, l)
                      in enumerate(zip(*hl)) if l not in hl[1][:i_]]
            ax.legend(*zip(*unique), loc="upper right")
 
    plt.suptitle(title, y=1.02, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
 
 
n_units_out = beta_rl.shape[1]
 
plot_beta_distributions(
    beta_rl[:, 0, :], true_betas[0, :],
    param_names, "Posterior Distributions — Household 0",
)
plot_beta_distributions(
    beta_rl[:, -1, :], true_betas[-1, :],
    param_names, f"Posterior Distributions — Household {n_units_out - 1}",
)
 
print("\nAll diagnostics complete.")